# IFC2x3 Duplex Architecture - COBie Data Transformation

This notebook processes IFC model JSON to add COBie values from an Excel reference table.
- **Input**: `JSON Whole Model/Ifc2x3_Duplex_Architecture.json` and COBie Excel lookup
- **Output**: `JSON_Edit/Ifc2x3_Duplex_Architecture.json` with COBie values added
- **Processed Items**: Walls, Windows, Doors


In [ ]:
from pathlib import Path
from shutil import copy2
import json
import pandas as pd
from IPython.display import display

# ============================================================
# SETUP & PATHS
# ============================================================

workspace_root = (Path.cwd() / '..').resolve()
source_json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
json_edit_dir = workspace_root / 'JSON_Edit'

# Verify files exist
assert source_json_path.exists(), f'JSON not found: {source_json_path}'
assert excel_path.exists(), f'Excel not found: {excel_path}'

json_edit_dir.mkdir(parents=True, exist_ok=True)
working_json_path = json_edit_dir / source_json_path.name

# Backup source if working copy doesn't exist
if not working_json_path.exists():
    copy2(source_json_path, working_json_path)

print(f'Source JSON: {source_json_path}')
print(f'Working JSON: {working_json_path}')
print(f'Excel Reference: {excel_path}')


In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_prop_value(properties, category, display_name):
    """Get a property value from items's Properties list."""
    if not isinstance(properties, list):
        return None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if (str(prop.get('category', '')).strip().upper() == category and 
            str(prop.get('displayName', '')).strip().upper() == display_name):
            return str(prop.get('value', '')).strip()
    return None


def is_type_match(item, target_type):
    """Check if item.Properties has Item/Type matching target_type."""
    properties = item.get('Properties', []) if isinstance(item, dict) else []
    if not isinstance(properties, list):
        return False
    
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        category = str(prop.get('category', '')).strip().lower()
        display_name = str(prop.get('displayName', '')).strip().lower()
        value = str(prop.get('value', '')).strip().upper()
        
        if category == 'item' and display_name == 'type' and value == target_type:
            return True
    return False


def extract_items_by_type(data, target_type):
    """Extract all items matching a specific IFC Type from source data."""
    rows = []
    for item in data:
        if not isinstance(item, dict):
            continue
        if is_type_match(item, target_type):
            rows.append({
                'GUID': str(item.get('ExternalId', '')).strip(),
                'Name': item.get('Name', ''),
                'DbId': item.get('DbId'),
                'Item.Type': target_type,
            })
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by=['Name', 'GUID'], kind='stable').reset_index(drop=True)
        df.insert(0, 'Count', df.index + 1)
    
    return df


def apply_cobie_value(target_item, cobie_value):
    """Apply COBie value to an item, returning (was_updated, before_value, after_value)."""
    properties = target_item.get('Properties')
    if not isinstance(properties, list):
        properties = []
        target_item['Properties'] = properties
    
    existing_cobie_prop = None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if (str(prop.get('category', '')).strip() == 'IFC' and 
            str(prop.get('displayName', '')).strip() == 'COBie'):
            existing_cobie_prop = prop
            break
    
    before_value = None
    if existing_cobie_prop is not None:
        before_value = str(existing_cobie_prop.get('value', '')).strip() or '(empty)'
        if before_value != cobie_value:
            existing_cobie_prop['value'] = cobie_value
            return True, before_value, cobie_value
        else:
            return False, before_value, cobie_value
    else:
        properties.append({
            'category': 'IFC',
            'displayName': 'COBie',
            'value': cobie_value,
        })
        return True, '(none)', cobie_value


In [ ]:
# ============================================================
# LOAD DATA & SEARCH EXCEL FOR COBIE MAPPINGS
# ============================================================

# Load source and working data
with source_json_path.open('r', encoding='utf-8') as f:
    source_data = json.load(f)

with working_json_path.open('r', encoding='utf-8') as f:
    working_data = json.load(f)

# Load Excel COBie reference table
ef_df = pd.read_excel(excel_path, sheet_name='EF', header=2)

# Define what to process: IFC type -> Excel title search term
items_to_process = [
    {'ifc_type': 'IFCWALL', 'excel_search': 'wall', 'display_name': 'Walls'},
    {'ifc_type': 'IFCDOOR', 'excel_search': 'door', 'display_name': 'Doors'},
    {'ifc_type': 'IFCWINDOW', 'excel_search': 'window', 'display_name': 'Windows'},
]

# Search Excel and build COBie mapping
cobie_mapping = {}
for config in items_to_process:
    ifc_type = config['ifc_type']
    search_term = config['excel_search']
    display_name = config['display_name']
    
    # Find rows in Excel where Title contains search term (case-insensitive)
    matched_rows = ef_df[ef_df['Title'].astype(str).str.contains(search_term, case=False, na=False)].copy()
    matched_rows = matched_rows.sort_values(by=['Title', 'Code'], kind='stable').reset_index(drop=True)
    
    if not matched_rows.empty:
        cobie_value = str(matched_rows.iloc[0]['COBie']).strip()
        cobie_mapping[ifc_type] = {
            'value': cobie_value,
            'excel_title_df': matched_rows,
            'count': len(matched_rows),
            'display_name': display_name
        }
        print(f'{display_name:10s} - COBie value found: {cobie_value}')
    else:
        print(f'{display_name:10s} - WARNING: No COBie mapping found in Excel!')

print(f'\nTotal items to process: {len(cobie_mapping)}')


In [ ]:
# ============================================================
# BEFORE CHECK: Extract items from source data (before processing)
# ============================================================

before_check = {}
for ifc_type, mapping in cobie_mapping.items():
    df = extract_items_by_type(source_data, ifc_type)
    before_check[ifc_type] = {
        'count': len(df),
        'dataframe': df,
        'display_name': mapping['display_name']
    }

# Summary table
before_summary_rows = []
for ifc_type, data in before_check.items():
    before_summary_rows.append({
        'Item Type': data['display_name'],
        'IFC Type': ifc_type,
        'Count in Source': data['count'],
        'COBie Value': cobie_mapping[ifc_type]['value'],
        'Excel Rows': cobie_mapping[ifc_type]['count']
    })

before_summary_df = pd.DataFrame(before_summary_rows)

print('=' * 80)
print('BEFORE PROCESSING CHECK')
print('=' * 80)
display(before_summary_df)

print('\nDetailed item lists:')
for ifc_type, data in before_check.items():
    print(f"\n{data['display_name']} ({ifc_type}):")
    if not data['dataframe'].empty:
        display(data['dataframe'][['Count', 'GUID', 'Name', 'DbId', 'Item.Type']])
    else:
        print(f"  No items found")


In [ ]:
# ============================================================
# APPLY COBIE VALUES (Consolidated Processing)
# ============================================================

# Build GUID lookup for working data
working_by_guid = {}
for item in working_data:
    if not isinstance(item, dict):
        continue
    guid = str(item.get('ExternalId', '')).strip()
    if guid:
        working_by_guid[guid] = item

# Process all item types and collect results
process_results = {}

for ifc_type, mapping in cobie_mapping.items():
    display_name = mapping['display_name']
    cobie_value = mapping['value']
    source_df = before_check[ifc_type]['dataframe']
    
    if source_df.empty:
        print(f'\nSkipping {display_name}: No items found in source data')
        continue
    
    print(f'\nProcessing {display_name} ({ifc_type})...')
    
    updated_count = 0
    added_count = 0
    missing_guid_count = 0
    before_values = []
    after_values = []
    
    for row in source_df.itertuples(index=False):
        target_item = working_by_guid.get(str(row.GUID).strip())
        if not isinstance(target_item, dict):
            missing_guid_count += 1
            before_values.append('(not found)')
            after_values.append('(not found)')
            continue
        
        was_updated, before_val, after_val = apply_cobie_value(target_item, cobie_value)
        
        before_values.append(before_val)
        after_values.append(after_val)
        
        if was_updated:
            updated_count += 1
            if before_val == '(none)':
                added_count += 1
    
    # Store results for after check
    process_results[ifc_type] = {
        'updated_count': updated_count,
        'added_count': added_count,
        'missing_guid_count': missing_guid_count,
        'total_items': len(source_df),
        'before_values': before_values,
        'after_values': after_values,
        'display_name': display_name
    }
    
    print(f'  Updated: {updated_count} | Added: {added_count} | Missing GUIDs: {missing_guid_count}')

# Write updated JSON
with working_json_path.open('w', encoding='utf-8') as f:
    json.dump(working_data, f, ensure_ascii=False, indent=2)

print(f'\n\n✓ Updated JSON written to: {working_json_path}')


In [ ]:
# ============================================================
# AFTER CHECK: Verify results and show before/after comparison
# ============================================================

print('=' * 80)
print('AFTER PROCESSING CHECK & SUMMARY')
print('=' * 80)

# Summary table of changes
after_summary_rows = []
total_updated = 0
total_added = 0
total_missing = 0

for ifc_type, results in process_results.items():
    after_summary_rows.append({
        'Item Type': results['display_name'],
        'IFC Type': ifc_type,
        'Total Items': results['total_items'],
        'Updated': results['updated_count'],
        'Added': results['added_count'],
        'Missing GUIDs': results['missing_guid_count'],
        'COBie Value': cobie_mapping[ifc_type]['value']
    })
    total_updated += results['updated_count']
    total_added += results['added_count']
    total_missing += results['missing_guid_count']

after_summary_df = pd.DataFrame(after_summary_rows)
display(after_summary_df)

# Overall statistics
print(f'\n{"OVERALL STATISTICS":^80}')
print(f'Total Items Processed: {sum(r["total_items"] for r in process_results.values())}')
print(f'Total Updated:         {total_updated}')
print(f'Total Added:           {total_added}')
print(f'Total Missing GUIDs:   {total_missing}')
print(f'Source JSON Records:   {len(source_data)}')
print(f'\nOutput Path: {working_json_path}')

# Detailed before/after values for each item type
print(f'\n{"DETAILED BEFORE/AFTER COMPARISON":^80}')

for ifc_type, results in process_results.items():
    display_name = results['display_name']
    source_df = before_check[ifc_type]['dataframe']
    
    if source_df.empty:
        continue
    
    # Create detailed output table
    detail_df = source_df[['Count', 'GUID', 'Name', 'DbId', 'Item.Type']].copy()
    detail_df['COBie Before'] = results['before_values']
    detail_df['COBie After'] = results['after_values']
    
    print(f'\n{display_name} ({ifc_type}) - Before/After Comparison:')
    display(detail_df)
    
    # Show unique before/after values
    before_unique = sorted(set(results['before_values']))
    after_unique = sorted(set(results['after_values']))
    print(f'  Before unique values: {before_unique}')
    print(f'  After unique values:  {after_unique}')

print(f'\n{"PROCESSING COMPLETE":^80}')
print(f'✓ All COBie values successfully applied and saved to: {working_json_path}')
